In [11]:
from pycelonis import get_celonis
import pandas as pd
import json
import time
from datetime import datetime, timezone, timedelta
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

In [2]:
team_url = ''
api_key = ''
key_type = 'USER_KEY'
data_pool_id = ''
data_job_id = ''

celonis = get_celonis(team_url, api_key, key_type)
pool = celonis.data_integration.get_data_pool(data_pool_id)
job = pool.get_job(data_job_id)

[2026-02-11 18:13:01,880] INFO: Initial connect successful! PyCelonis Version: 2.14.1
[2026-02-11 18:13:02,108] INFO: `ml-workbench` permissions: ['CREATE_APPS', 'USE_ALL_APPS', 'MANAGE_ALL_APPS', 'CREATE_WORKSPACES', 'MANAGE_ALL_WORKSPACES', 'VIEW_CONFIGURATION']
[2026-02-11 18:13:02,109] INFO: `team` permissions: ['MANAGE_AUDIT_LOGS', 'MANAGE_SSO_SETTINGS', 'USE_AUDIT_LOGS_API', 'TEAM_TO_TEAM_COPY', 'MANAGE_ADOPTION_VIEWS', 'MANAGE_GENERAL_SETTINGS', 'MANAGE_GROUPS', 'MANAGE_APPLICATIONS', 'MANAGE_ON_PREM_CLIENTS', 'USE_STUDIO_ADOPTION_API', 'MANAGE_LOGIN_HISTORY', 'MANAGE_LICENSE_SETTINGS', 'USE_LOGIN_HISTORY_API', 'USE_USER_GROUP_INFO_API', 'MANAGE_MEMBERS', 'MANAGE_UPLINK_INTEGRATIONS', 'MANAGE_PERMISSIONS', 'MANAGE_ADMIN_NOTIFICATIONS', 'MANAGE_DOWNLOAD_PORTAL', 'IMPORT_MEMBERS']
[2026-02-11 18:13:02,109] INFO: `process-repository` permissions: ['CREATE_AND_MODIFY_CATEGORIES', 'USE_CATEGORIES', 'DELETE_EXISTING_CATEGORIES', 'MODIFY_EXISTING_CATEGORIES']
[2026-02-11 18:13:02,109] 

In [3]:
dt = datetime.fromtimestamp(1770821783640 / 1000, tz=timezone.utc)
iso_string = dt.replace(microsecond=0).isoformat()
iso_string

'2026-02-11T14:56:23+00:00'

In [68]:
job_logs = celonis.client.request(url=f'{team_url}/integration/api/pools/{data_pool_id}/logs/{data_job_id}/executions?limit=25&page=0', method = 'GET').json()

def convert_time(timestamp):
    dt = str(datetime.fromtimestamp(timestamp / 1000, tz=timezone.utc).replace(microsecond=0).isoformat())
    dt = str(pd.to_datetime(dt, utc=True).tz_convert("America/New_York").strftime("%Y-%m-%d %H:%M:%S"))
    return dt

def get_runtime(start, end):
    runtime = str(timedelta(milliseconds=end - start)).split('.')[0]
    return runtime
    
execution_dict = {}
for log in job_logs['executionItems']:
    if log['status'] in ['QUEUED', 'RUNNING']:
        if log['startDate'] == None:
            ts = time.time()
            starttime = datetime.fromtimestamp(ts).replace(microsecond=0).isoformat()
        else:
            starttime = convert_time(log['startDate'])
        endtime = None
        runtime = None
    else:
        starttime = convert_time(log['startDate'])
        endtime = convert_time(log['endDate'])
        runtime = get_runtime(log['startDate'], log['endDate'])
        
    schedule_name = log['schedulingName']
    executions = celonis.client.request(url=f'{team_url}integration/api/pools/{data_pool_id}/logs/executions?executionId={log["executionId"]}&type=TASK&id={data_job_id}', method = 'GET').json()
    
    execution_dict[schedule_name + ' | ' + starttime] = {}
    execution_dict[schedule_name + ' | ' + starttime]['startDate'] = starttime
    execution_dict[schedule_name + ' | ' + starttime]['endDate'] = endtime
    execution_dict[schedule_name + ' | ' + starttime]['runtime'] = runtime
    execution_dict[schedule_name + ' | ' + starttime]['status'] = log['status']
    # execution_dict[schedule_name + ' | ' + starttime]['metadata'] = executions
    execution_dict[schedule_name + ' | ' + starttime]['transformations'] = {}
    
    for trans in executions:
        execution_dict[schedule_name + ' | ' + starttime]['transformations'][trans['name']] = {}
        if trans['status'] in ['QUEUED', 'RUNNING']:
            trans_runtime = None
        else:
            trans_runtime = get_runtime(trans['startDate'], trans['endDate'])
            trans_starttime = convert_time(trans['startDate'])
            trans_endtime = convert_time(trans['endDate'])
            
        execution_dict[schedule_name + ' | ' + starttime]['transformations'][trans['name']]['runtime'] = trans_runtime
        execution_dict[schedule_name + ' | ' + starttime]['transformations'][trans['name']]['startDate'] = trans_starttime
        execution_dict[schedule_name + ' | ' + starttime]['transformations'][trans['name']]['endDate'] = trans_endtime 
        execution_dict[schedule_name + ' | ' + starttime]['transformations'][trans['name']]['status'] = trans['status']

In [81]:
def transformation_variability_report(execution_dict):
    runtimes = defaultdict(list)
    total_runs = len(execution_dict)

    for run_key, run_payload in execution_dict.items():
        # extract high-level start time from run key
        # assumes format: "... | <timestamp>"
        run_start = run_key.split(" | ")[-1].split('+')[0]

        for t_name, t_payload in run_payload.get("transformations", {}).items():
            rt = t_payload.get("runtime")
            if not rt:
                continue

            secs = pd.to_timedelta(rt).total_seconds()

            if secs > 0:  # exclude "did not run"
                runtimes[t_name].append((secs, run_start))

    rows = []

    for t_name, sec_key_pairs in runtimes.items():
        secs = np.array([x[0] for x in sec_key_pairs], dtype=float)
        starts = [x[1] for x in sec_key_pairs]

        mean = round(secs.mean())
        median = float(np.median(secs))
        maximum = float(secs.max())
        minimum = float(secs.min())
        p95 = round(float(np.percentile(secs, 95)))
        std = round(float(secs.std(ddof=1)) if len(secs) > 1 else 0.0)
        cv = round(std / mean if mean > 0 else 0.0)

        max_idx = int(np.argmax(secs))
        min_idx = int(np.argmin(secs))

        rows.append({
            "TRANSFORMATION": t_name,
            "TOTAL_RUNS": total_runs,
            "SUCCESSFUL_RUNS": len(secs),
            "RUN_RATE": len(secs) / total_runs,

            "MEAN_SEC": mean,
            "MEDIAN_SEC": median,

            "MIN_SEC": minimum,
            "MIN_RUN_START": starts[min_idx],

            "MAX_SEC": maximum,
            "MAX_RUN_START": starts[max_idx],

            "P95_SEC": p95,
            "STD_SEC": std,
            "CV": cv,
        })

    df = (
        pd.DataFrame(rows)
        .sort_values("MEAN_SEC", ascending=False)
        .reset_index(drop=True)
    )

    return df

trans_df = transformation_variability_report(execution_dict)
trans_df.to_csv("transformation_log_report.csv", index=False)

In [82]:
def runs_summary(execution_dict):
    rows = []
    for run_key, run_payload in execution_dict.items():
        start = run_payload.get("startDate")
        end = run_payload.get("endDate")
        rt = run_payload.get("runtime")
        status = run_payload.get("status")

        rows.append({
            "RUN_KEY": run_key,
            "STARTDATE": start,
            "ENDDATE": end,
            "RUNTIME": rt,
            "STATUS": status,
            # optional: normalize to seconds + HH:MM:SS if runtime exists
            "RUNTIME_SECONDS": pd.to_timedelta(rt).total_seconds() if rt else None,
            "RUNTIME_HHMMSS": str(pd.to_timedelta(rt)).split(".")[0] if rt else None,
        })

    df = pd.DataFrame(rows)

    # optional: make start/end actual datetimes (keeps timezone if present)
    df["STARTDATE"] = pd.to_datetime(df["STARTDATE"], errors="coerce")
    df["ENDDATE"] = pd.to_datetime(df["ENDDATE"], errors="coerce")

    # sort by start time
    df = df.sort_values("STARTDATE", ascending=False).reset_index(drop=True)
    return df

log_df = runs_summary(execution_dict)
log_df.to_csv('schedule_log_report.csv')

In [83]:
def single_run_transformation_report(run_payload, sort_by="	STARTDATE", descending=True):
    rows = []

    transformations = run_payload.get("transformations", {})
    for t_name, t_payload in transformations.items():
        rt = t_payload.get("runtime")
        start = t_payload.get("startDate")
        end = t_payload.get("endDate")
        status = t_payload.get("status")

        rt_seconds = pd.to_timedelta(rt).total_seconds() if rt else None

        rows.append({
            "TRANSFORMATION": t_name,
            "STATUS": status,
            "STARTDATE": pd.to_datetime(start, errors="coerce"),
            "ENDDATE": pd.to_datetime(end, errors="coerce"),
            "RUNTIME": rt,
            "RUNTIME_SECONDS": rt_seconds,
            "RUNTIME_HHMMSS": str(pd.to_timedelta(rt_seconds, unit="s")).split(".")[0] if rt_seconds is not None else None,
        })

    df = pd.DataFrame(rows)

    # sort
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=not descending)

    df = df.reset_index(drop=True)
    return df

In [84]:
# For long run
run_key = "Zendesk Delta Load + OCDM | 2026-02-11 03:00:49"
run_payload = execution_dict[run_key]

long_run_df = single_run_transformation_report(run_payload, sort_by="RUNTIME_SECONDS", descending=True)
long_run_df.to_csv('long_run_log.csv')

In [85]:
# For short run
run_key = "Zendesk Delta Load + OCDM | 2026-02-10 18:37:54"
run_payload = execution_dict[run_key]

short_run_df = single_run_transformation_report(run_payload, sort_by="RUNTIME_SECONDS", descending=True)
short_run_df.to_csv('short_run_log.csv')